In [1]:
import tensorflow as tf
import numpy as np

In [2]:
path = tf.keras.utils.get_file(
    'shakespeare.txt',
    'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
)

text = open(path, 'r', encoding='utf-8').read().lower()

print("Total characters:", len(text))
print(text[:500])

1115394/1115394 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Total characters: 1115394
first citizen:
before we proceed any further, hear me speak.

all:
speak, speak.

first citizen:
you are all resolved rather to die than to famish?

all:
resolved. resolved.

first citizen:
first, you know caius marcius is chief enemy to the people.

all:
we know't, we know't.

first citizen:
let us kill him, and we'll have corn at our own price.
is't a verdict?

all:
no more talking on't; let it be done: away, away!

second citizen:
one word, good citizens.

first citizen:
we are accounted poor


In [3]:
chars = sorted(set(text))

c2i = {c: i for i, c in enumerate(chars)}
i2c = {i: c for i, c in enumerate(chars)}

print("Unique characters:", len(chars))
print(chars)

Unique characters: 39
['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [4]:
seq_len = 40
X = []
y = []

for i in range(len(text) - seq_len):
    input_seq = text[i:i + seq_len]
    target_char = text[i + seq_len]

    X.append([c2i[c] for c in input_seq])
    y.append(c2i[target_char])

X = np.array(X)
y = np.array(y)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (1115354, 40)
y shape: (1115354,)


In [5]:
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(len(chars), 64),
    tf.keras.layers.LSTM(128),
    tf.keras.layers.Dense(len(chars), activation='softmax')
])

model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam'
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm (LSTM)                          │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [6]:
history = model.fit(
    X,
    y,
    batch_size=128,
    epochs=1
)

8714/8714 ━━━━━━━━━━━━━━━━━━━━ 480s 55ms/step - loss: 1.9094


In [7]:
def generate(seed, length=300):
    seq = [c2i[c] for c in seed.lower()]

    for _ in range(length):
        inp = np.array(seq[-seq_len:]).reshape(1, -1)
        pred = model.predict(inp, verbose=0)[0]
        next_idx = np.random.choice(len(pred), p=pred)
        seq.append(next_idx)

    return seed + ''.join(i2c[i] for i in seq[len(seed):])

In [8]:
print("\nGenerated Text:\n")

print(generate(
    "shall i compare thee to a summer's day?\n"
))


Generated Text:

shall i compare thee to a summer's day?
hather badn's in the farthithy thee, with
of him lows of for she copkick frear in llanco.


jrothy:
if, and the deid my thin unrother musckns.
is you. comsence, a, i'll and not he kign,
begnenance, but hasging he pear him he will;
that, with then, if for all usticor.

fartithing:
you were i am the a


In [9]:
print("TensorFlow version:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices('GPU'))

TensorFlow version: 2.22.0-rc0
GPUs: []
